In [96]:
import pandas as pd
import torch
import time
import json
import os

In [ ]:
df = pd.read_csv('../booksummaries/book_data_clean.csv')
df.head()

,id,title,author,pub_date,genres,summary
0,620,Animal Farm,George Orwell,1945-08-17,"[""Roman à clef"", ""Satire"", ""Children's literat...","Old Major, the old boar on the Manor Farm, ca..."
1,843,A Clockwork Orange,Anthony Burgess,1962-01-01,"[""Science Fiction"", ""Novella"", ""Speculative fi...","Alex, a teenager living in near-future Englan..."
2,986,The Plague,Albert Camus,1947-01-01,"[""Existentialism"", ""Fiction"", ""Absurdist ficti...",The text of The Plague is divided into five p...
3,1756,An Enquiry Concerning Human Understanding,David Hume,NaN,[],The argument of the Enquiry proceeds by a ser...
4,2080,A Fire Upon the Deep,Vernor Vinge,NaN,"[""Hard science fiction"", ""Science Fiction"", ""S...",The novel posits that space around the Milky ...


In [98]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using GPU') if device == 'cuda' else print('Using CPU')

cols = [c for c in ['title', 'author', 'genres', 'summary'] if c in df.columns]
text = df[cols].fillna('').astype(str).agg(' | '.join, axis=1)

corpus = b'\n'.join(t.encode('utf-8') for t in text)

# print(list(corpus[:200]))

Using GPU


In [99]:
tokens = torch.tensor(list(corpus), dtype=torch.int32, device=device)

vocab = {i:bytes([i]) for i in range(256)}
merges= []
next_id = 256

special_tokens = ["[PAD]", "[CLS]", "[SEP]", "[BOS]", "[EOS]", "[QUERY]", "[BOOK]", "[SUM]"]
special_token_ids = {}
for tok in special_tokens:
    special_token_ids[tok] = next_id
    vocab[next_id] = tok.encode('utf-8')
    next_id += 1

vocab_size = 16000
min_freq = 2
max_merges = vocab_size - next_id 

In [100]:
@torch.no_grad()
def get_pair_counts(tokens: torch.Tensor):
    if tokens.numel() < 2:
        return None
    a = tokens[:-1].to(torch.int64)
    b = tokens[1:].to(torch.int64)
    pairs = (a<<32) | b
    sorted_pairs, _ = torch.sort(pairs)
    unique, counts = torch.unique_consecutive(sorted_pairs, return_counts=True)
    if unique.numel() == 0:
        return None
    
    best_idx = torch.argmax(counts)
    best_key = unique[best_idx].item()
    best_pair = ((best_key >> 32) & 0xFFFFFFFF, best_key & 0xFFFFFFFF)
    best_freq = counts[best_idx].item()
    return (best_pair, best_freq)

In [101]:
@torch.no_grad()
def merge(tokens:torch.Tensor, pair, new_id):
    a,b = pair
    if tokens.numel() < 2:
        return tokens
    first = tokens[:-1] == a
    second = tokens[1:] == b
    match = first & second
    if not torch.any(match):
        return tokens
    idx = torch.nonzero(match, as_tuple=False).squeeze(1)
    tokens[idx] = new_id

    mask = torch.ones(tokens.size(0), dtype=torch.bool, device=tokens.device)
    mask[idx + 1] = False
    return tokens[mask]

In [102]:
for i in range(max_merges):
    res = get_pair_counts(tokens)
    if res is None:
        break
    best_pair, best_freq = res
    before = tokens.numel()
    tokens = merge(tokens, best_pair, next_id)
    if tokens.numel() == before:
        break
    vocab[next_id] = vocab[best_pair[0]] + vocab[best_pair[1]]
    merges.append((best_pair, next_id))
    next_id+=1
    if next_id >= vocab_size or tokens.numel() < 2:
        break

In [104]:
os.makedirs('bpe_model', exist_ok=True)
with open('bpe_model/merges.json', 'w', encoding='utf-8') as f:
    json.dump({'merges': [{'pair': p, 'id': i} for p, i in merges], 'special_tokens': special_token_ids}, f)
with open('bpe_model/vocab.json', 'w', encoding='utf-8') as f:
    json.dump({str(k): v.hex() for k,v in vocab.items()}, f)
print('Saved optimized model to bpe_model/.')

Saved optimized model to bpe_model/.


In [ ]:
def decode(token_ids):
    return b''.join(vocab[t] for t in token_ids).decode('utf-8', errors='replace')

# Precompute helper maps for faster encoding (pair -> new_id, pair -> rank)
# pair_to_newid = {pair: new_id for pair, new_id in merges}
# pair_rank = {pair: i for i, (pair, new_id) in enumerate(merges)}  # lower rank = earlier merge

# INF_RANK = 10**12

# def _get_pairs(seq):
#     # Return set of adjacent pairs in sequence
#     return {(seq[i], seq[i+1]) for i in range(len(seq)-1)}

# def encode(text: str):
#     # Byte initialize
#     seq = list(text.encode('utf-8'))
#     if len(seq) < 2 or not merges:
#         return seq
#     pairs = _get_pairs(seq)
#     while True:
#         # Select best (lowest rank) pair present that exists in pair_rank
#         best_pair = None
#         best_rank = INF_RANK
#         for p in pairs:
#             r = pair_rank.get(p, INF_RANK)
#             if r < best_rank:
#                 best_rank = r
#                 best_pair = p
#         if best_pair is None or best_rank == INF_RANK:
#             break  # no mergeable pair
#         new_id = pair_to_newid[best_pair]
#         a, b = best_pair
#         new_seq = []
#         i = 0
#         L = len(seq)
#         while i < L:
#             if i < L - 1 and seq[i] == a and seq[i+1] == b:
#                 new_seq.append(new_id)
#                 i += 2
#             else:
#                 new_seq.append(seq[i])
#                 i += 1
#         seq = new_seq
#         if len(seq) < 2:
#             break
#         pairs = _get_pairs(seq)
#     return seq

def encode(text: str):
    ids = list(text.encode('utf-8'))
    for (a,b), nid in merges:
        out = []
        i = 0
        while i < len(ids):
            if i < len(ids)-1 and ids[i] == a and ids[i+1] == b:
                out.append(nid)
                i += 2
            else:
                out.append(ids[i])
                i += 1
        ids = out
    return ids


In [122]:
ec = encode('''“Of course, I’ve been meaning lately to go to Razumihin’s to ask for work, to ask him to get me lessons or something...” Raskolnikov thought, “but what help can he be to me now? Suppose he gets me lessons, suppose he shares his last farthing with me, if he has any farthings, so that I could get some boots and make myself tidy enough to give lessons... hm... Well and what then? What shall I do with the few coppers I earn? That’s not what I want now. It’s really absurd for me to go to Razumihin....”

The question why he was now going to Razumihin agitated him even more than he was himself aware; he kept uneasily seeking for some sinister significance in this apparently ordinary action.

“Could I have expected to set it all straight and to find a way out by means of Razumihin alone?” he asked himself in perplexity.

He pondered and rubbed his forehead, and, strange to say, after long musing, suddenly, as if it were spontaneously and by chance, a fantastic thought came into his head.

“Hm... to Razumihin’s,” he said all at once, calmly, as though he had reached a final determination. “I shall go to Razumihin’s of course, but... not now. I shall go to him... on the next day after It, when It will be over and everything will begin afresh....”

“Finish her off,” shouted Mikolka and he leapt beside himself, out of the cart. Several young men, also flushed with drink, seized anything they could come across—whips, sticks, poles, and ran to the dying mare. Mikolka stood on one side and began dealing random blows with the crowbar. The mare stretched out her head, drew a long breath and died.
                 ''')
ec

[1597,
 11757,
 12438,
 73,
 645,
 477,
 575,
 9058,
 715,
 650,
 281,
 2695,
 82,
 1102,
 455,
 105,
 4504,
 645,
 464,
 2287,
 360,
 6091,
 281,
 2287,
 1041,
 883,
 1726,
 2794,
 2314,
 410,
 13926,
 1599,
 5078,
 82,
 1742,
 314,
 110,
 1631,
 4645,
 2336,
 659,
 1597,
 416,
 837,
 989,
 713,
 296,
 435,
 281,
 1726,
 2352,
 2928,
 10289,
 706,
 264,
 296,
 1318,
 1726,
 2794,
 12079,
 4633,
 264,
 296,
 2977,
 1223,
 1325,
 2585,
 957,
 339,
 109,
 368,
 970,
 1377,
 928,
 2585,
 15232,
 5154,
 1239,
 1375,
 883,
 849,
 635,
 2748,
 1071,
 3966,
 683,
 292,
 3064,
 4598,
 1826,
 14274,
 115,
 5802,
 13316,
 1599,
 505,
 737,
 282,
 837,
 5593,
 2928,
 7819,
 6149,
 1239,
 1043,
 577,
 1487,
 2980,
 112,
 413,
 1239,
 5679,
 2928,
 905,
 302,
 746,
 463,
 837,
 1239,
 2350,
 2352,
 496,
 116,
 746,
 2756,
 15183,
 269,
 360,
 109,
 490,
 2695,
 82,
 1102,
 455,
 105,
 4504,
 1599,
 1599,
 1589,
 10,
 4750,
 5101,
 2831,
 1965,
 902,
 2910,
 82,
 1102,
 455,
 105,
 104,
 300,
 342,


In [123]:
print(decode(ec))

“Of course, I’ve been meaning lately to go to Razumihin’s to ask for work, to ask him to get me lessons or something...” Raskolnikov thought, “but what help can he be to me now? Suppose he gets me lessons, suppose he shares his last farthing with me, if he has any farthings, so that I could get some boots and make myself tidy enough to give lessons... hm... Well and what then? What shall I do with the few coppers I earn? That’s not what I want now. It’s really absurd for me to go to Razumihin....”

The question why he was now going to Razumihin agitated him even more than he was himself aware; he kept uneasily seeking for some sinister significance in this apparently ordinary action.

“Could I have expected to set it all straight and to find a way out by means of Razumihin alone?” he asked himself in perplexity.

He pondered and rubbed his forehead, and, strange to say, after long musing, suddenly, as if it were spontaneously and by chance, a fantastic thought came into his head.

“Hm.